# Hybrid Search and Query Routing
Companion notebook for the **Hybrid Search and Query Routing** section of Chapter 10.

Keyword search wins on exact terms (product names, error codes, IDs). Vector search wins on meaning (paraphrases, synonyms). A common production pattern runs both and **blends the scores**.

This is the book's `hybrid_score`, unchanged:

In [1]:
def hybrid_score(bm25, vector_sim, w_keyword=0.4, w_vector=0.6):
    return w_keyword * bm25 + w_vector * vector_sim

In [2]:
# BM25 and vector scores are on different scales -- normalize to [0, 1] before blending.
candidates = [
    {"chunk_id": "c1", "bm25": 0.92, "vector_sim": 0.41},  # strong exact match, weak semantic match
    {"chunk_id": "c2", "bm25": 0.10, "vector_sim": 0.88},  # weak exact match, strong paraphrase match
    {"chunk_id": "c3", "bm25": 0.55, "vector_sim": 0.55},  # middling on both
]

for c in candidates:
    blended = hybrid_score(c["bm25"], c["vector_sim"])
    print(f"{c['chunk_id']}: bm25={c['bm25']:.2f} vector={c['vector_sim']:.2f} -> hybrid={blended:.3f}")

# For exact-entity queries, weight keyword higher; for broad "how do I" questions, weight vector higher.
print("\nweighted toward keyword (error-code style query):")
for c in candidates:
    print(f"  {c['chunk_id']}: {hybrid_score(c['bm25'], c['vector_sim'], w_keyword=0.8, w_vector=0.2):.3f}")

c1: bm25=0.92 vector=0.41 -> hybrid=0.614
c2: bm25=0.10 vector=0.88 -> hybrid=0.568
c3: bm25=0.55 vector=0.55 -> hybrid=0.550

weighted toward keyword (error-code style query):
  c1: 0.818
  c2: 0.256
  c3: 0.550


This is the book's `route_query`, unchanged -- decides keyword-only, vector-only, hybrid, or no retrieval at all:

In [3]:
def route_query(q: str):
    if q.lower() in {"hi", "hello", "thanks"}:
        return "no_retrieval"
    if any(token.isupper() and any(ch.isdigit() for ch in token) for token in q.split()):
        return "keyword"
    if len(q.split()) > 8:
        return "hybrid"
    return "vector"

In [4]:
test_queries = [
    "hello",
    "What does error E1027 mean?",
    "How do I get my money back after a purchase?",
    "What is our refund window for enterprise plans and does it change for annual contracts?",
]

for q in test_queries:
    print(f"{route_query(q):12s} <- {q!r}")

no_retrieval <- 'hello'
keyword      <- 'What does error E1027 mean?'
hybrid       <- 'How do I get my money back after a purchase?'
hybrid       <- 'What is our refund window for enterprise plans and does it change for annual contracts?'
